# Persona Consistency Ablation Results

This notebook analyzes the 100-agent OSS-120B ablation runs. It records the ablation setup, the metrics used for comparison, and the expected hypothesis before final interpretation.

The simulations may still be running. The notebook points only to the real ablation output folders and can run on partial outputs; rerun it after all four simulations complete before interpreting results.


In [1]:
from __future__ import annotations

from pathlib import Path
import contextlib
import io
import sys

import pandas as pd

ROOT = Path.cwd().resolve()
if ROOT.name == "analysis":
    ROOT = ROOT.parent
ANALYSIS_DIR = ROOT / "analysis"
for path in [ROOT, ANALYSIS_DIR]:
    if str(path) not in sys.path:
        sys.path.insert(0, str(path))

from evals_persona_consistency_ablations import evaluate_models
from utils import load_actual_reference_frames, load_simulation_run_dict
from evals import build_pattern_metrics_table, build_trip_metrics_table, build_chain_metrics_table

pd.set_option("display.max_columns", 80)
pd.set_option("display.width", 160)

SEED = 42
ANCHOR_MIN_MINUTES = 120
CONTROL_SAMPLES_PER_AGENT_DAY = 20
TARGET_TAXONOMY = "legacy6"
POPULATION_DAY_SELECTOR = "random"


## Ablation Setups

The four setups form a cumulative ladder. We use the existing runtime switches without changing simulation code.


In [2]:
ABLATION_RUNS = {
    "Reactive Only": "simulation_outputs/SF_100_oss_120b_ablation_reactive_only",
    "Planner Only": "simulation_outputs/SF_100_oss_120b_ablation_planner_only",
    "Planner + Memory": "simulation_outputs/SF_100_oss_120b_ablation_planner_memory",
    "Full Stack": "simulation_outputs/SF_100_oss_120b_ablation_full_stack",
}
INPUT_FOLDERS = {
    name: "Inputs/SF_agents_100" for name in ABLATION_RUNS
}

ABLATION_DESIGN = pd.DataFrame([
    {
        "setup": "Reactive Only",
        "config": "input_GABM_SERVER_ablation_reactive_only.yaml",
        "use_day_planner": False,
        "memory_ablation_mode": "no_memory",
        "behavioral_interpretation": "No daily plan and no retrieved memory context.",
        "output_folder": ABLATION_RUNS["Reactive Only"],
    },
    {
        "setup": "Planner Only",
        "config": "input_GABM_SERVER_ablation_planner_only.yaml",
        "use_day_planner": True,
        "memory_ablation_mode": "no_memory",
        "behavioral_interpretation": "Daily plan enabled; no retrieved memory context.",
        "output_folder": ABLATION_RUNS["Planner Only"],
    },
    {
        "setup": "Planner + Memory",
        "config": "input_GABM_SERVER_ablation_planner_memory.yaml",
        "use_day_planner": True,
        "memory_ablation_mode": "no_reflection",
        "behavioral_interpretation": "Daily plan plus episodic memory retrieval; no reflection summaries.",
        "output_folder": ABLATION_RUNS["Planner + Memory"],
    },
    {
        "setup": "Full Stack",
        "config": "input_GABM_SERVER.yaml",
        "use_day_planner": True,
        "memory_ablation_mode": "baseline",
        "behavioral_interpretation": "Daily plan, episodic memory retrieval, and reflection summaries.",
        "output_folder": ABLATION_RUNS["Full Stack"],
    },
])

RUN_FOLDERS = ABLATION_RUNS
ABLATION_DESIGN


,setup,config,use_day_planner,memory_ablation_mode,behavioral_interpretation,output_folder
0,Reactive Only,input_GABM_SERVER_ablation_reactive_only.yaml,False,no_memory,No daily plan and no retrieved memory context.,simulation_outputs/SF_100_oss_120b_ablation_re...
1,Planner Only,input_GABM_SERVER_ablation_planner_only.yaml,True,no_memory,Daily plan enabled; no retrieved memory context.,simulation_outputs/SF_100_oss_120b_ablation_pl...
2,Planner + Memory,input_GABM_SERVER_ablation_planner_memory.yaml,True,no_reflection,Daily plan plus episodic memory retrieval; no ...,simulation_outputs/SF_100_oss_120b_ablation_pl...
3,Full Stack,input_GABM_SERVER.yaml,True,baseline,"Daily plan, episodic memory retrieval, and ref...",simulation_outputs/SF_100_oss_120b_ablation_fu...


## Metrics

We report only the metrics needed for the ablation claim.

**Persona-continuity metrics** reuse `analysis/evals_persona_consistency_ablations.py` and use all seven simulated days:

- worker role-anchor adherence: higher is better
- student role-anchor adherence: higher is better
- worker anchor start-time std: lower is better
- student anchor start-time std: lower is better
- weekday routine continuity: higher is better
- weekday routine similarity delta: higher is better

**NHTS-style population metrics** reuse the existing evaluation helpers and use one random simulated day per agent (`POPULATION_DAY_SELECTOR = "random"), because NHTS is a diary-style person-day reference:

- average locations traveled per agent-day and signed delta vs actual SF: smaller absolute delta is better
- chain weight overlap with actual SF chains: higher is better
- first-order transition L2/Frobenius difference vs actual SF: lower is better

The population metrics currently use `TARGET_TAXONOMY = "legacy6"` to match the existing paper-facing NHTS-style evaluation tables. The persona-continuity metrics use the active runtime activity codes internally.


## Actual Run Folders

These are the completed 100-agent ablation output folders used in the scorecard below. The `activity_log_rows` and `observed_dates` columns verify that each run covers the seven-day simulation window.


In [3]:
def _activity_log_coverage(run_folder: str) -> dict:
    folder = ROOT / run_folder
    files = sorted(folder.glob("activity_log_rank*.csv")) if folder.exists() else []
    rows = 0
    dates = set()
    for path in files:
        try:
            frame = pd.read_csv(path, usecols=["date"])
        except Exception:
            continue
        rows += int(frame.shape[0])
        dates.update(str(value) for value in frame["date"].dropna().unique())
    return {
        "folder_exists": folder.exists(),
        "activity_log_files": len(files),
        "activity_log_rows": rows,
        "observed_dates": len(dates),
        "date_range": " to ".join([min(dates), max(dates)]) if dates else "",
    }

active_runs = pd.DataFrame([
    {
        "setup": setup,
        "run_folder": folder,
        "input_folder": INPUT_FOLDERS[setup],
        **_activity_log_coverage(folder),
    }
    for setup, folder in RUN_FOLDERS.items()
])
active_runs


,setup,run_folder,input_folder,folder_exists,activity_log_files,activity_log_rows,observed_dates,date_range
0,Reactive Only,simulation_outputs/SF_100_oss_120b_ablation_re...,Inputs/SF_agents_100,True,1,2944,7,2025-09-08 to 2025-09-14
1,Planner Only,simulation_outputs/SF_100_oss_120b_ablation_pl...,Inputs/SF_agents_100,True,1,3883,7,2025-09-08 to 2025-09-14
2,Planner + Memory,simulation_outputs/SF_100_oss_120b_ablation_pl...,Inputs/SF_agents_100,True,1,3712,7,2025-09-08 to 2025-09-14
3,Full Stack,simulation_outputs/SF_100_oss_120b_ablation_fu...,Inputs/SF_agents_100,True,1,3662,7,2025-09-08 to 2025-09-14


In [4]:
missing = active_runs.loc[~active_runs["folder_exists"], "run_folder"].tolist()
if missing:
    raise FileNotFoundError(
        "Missing real ablation run folders: " + ", ".join(missing)
    )


## Persona-Continuity Metrics

This table is the compact persona-metric view. It intentionally omits per-agent detail; the per-agent inspection notebook already handles that level of analysis.


In [5]:
persona_result = evaluate_models(
    RUN_FOLDERS,
    INPUT_FOLDERS,
    anchor_min_minutes=ANCHOR_MIN_MINUTES,
    seed=SEED,
    control_samples_per_agent_day=CONTROL_SAMPLES_PER_AGENT_DAY,
)

persona_cols = [
    "model",
    "worker_role_anchor_adherence_mean",
    "student_role_anchor_adherence_mean",
    "worker_anchor_start_std_median_minutes",
    "student_anchor_start_std_median_minutes",
    "weekday_routine_continuity_mean",
    "weekday_routine_similarity_delta_mean",
]
persona_summary = (
    persona_result.summary[persona_cols]
    .rename(columns={"model": "setup"})
    .round(4)
)
persona_summary


,setup,worker_role_anchor_adherence_mean,student_role_anchor_adherence_mean,worker_anchor_start_std_median_minutes,student_anchor_start_std_median_minutes,weekday_routine_continuity_mean,weekday_routine_similarity_delta_mean
0,Reactive Only,1.0000,0.9048,25.1496,20.9162,0.6838,0.0241
1,Planner Only,0.9959,0.8000,47.7755,45.6886,0.6107,0.0311
2,Planner + Memory,1.0000,0.7571,34.7131,32.2878,0.6508,0.0549
3,Full Stack,1.0000,0.7286,20.3101,32.1113,0.7217,0.1186


## NHTS-Style Population Metrics

This table reports one activity-chain metric and one first-order transition metric. These are the population-level complements to the persona-continuity metrics.


In [6]:
with contextlib.redirect_stdout(io.StringIO()):
    reference_frames = load_actual_reference_frames(target_taxonomy=TARGET_TAXONOMY)
    model_frames = load_simulation_run_dict(
        RUN_FOLDERS,
        day_selector=POPULATION_DAY_SELECTOR,
        seed=SEED,
        run_input_folders=INPUT_FOLDERS,
        target_taxonomy=TARGET_TAXONOMY,
    )

pattern_metrics = build_pattern_metrics_table(
    actual_sf_df=reference_frames["actual_sf"],
    model_frames=model_frames,
)
trip_metrics = build_trip_metrics_table(
    actual_sf_df=reference_frames["actual_sf"],
    model_frames=model_frames,
)
chain_metrics = build_chain_metrics_table(
    actual_sf_df=reference_frames["actual_sf"],
    actual_all_df=reference_frames["actual_all"],
    model_frames=model_frames,
)

actual_location_mean = pd.to_numeric(reference_frames["actual_sf"]["location"], errors="coerce").dropna().mean()
location_mean = (
    pattern_metrics.loc[pattern_metrics["metric"] == "location_count_mean", ["model", "value"]]
    .rename(columns={"model": "setup", "value": "avg_locations_traveled"})
)
location_mean["actual_sf_avg_locations_traveled"] = actual_location_mean
location_mean["avg_locations_delta_vs_actual"] = (
    location_mean["avg_locations_traveled"] - location_mean["actual_sf_avg_locations_traveled"]
)

transition_l2 = (
    trip_metrics.loc[trip_metrics["metric"] == "transition_matrix_l2_diff_vs_actual", ["model", "value"]]
    .rename(columns={"model": "setup", "value": "transition_l2_diff_vs_actual"})
)
chain_overlap = (
    chain_metrics.loc[chain_metrics["metric"] == "chain_weight_overlap_actual_sf_generated", ["model", "value"]]
    .rename(columns={"model": "setup", "value": "chain_weight_overlap_actual_sf_generated"})
)
population_summary = (
    location_mean
    .merge(chain_overlap, on="setup", how="outer")
    .merge(transition_l2, on="setup", how="outer")
)
population_summary.insert(1, "day_selector", POPULATION_DAY_SELECTOR)
population_summary.insert(2, "taxonomy", TARGET_TAXONOMY)
population_summary = population_summary.round(4)
population_summary


,setup,day_selector,taxonomy,avg_locations_traveled,actual_sf_avg_locations_traveled,avg_locations_delta_vs_actual,chain_weight_overlap_actual_sf_generated,transition_l2_diff_vs_actual
0,Full Stack,random,legacy6,5.36,5.3727,-0.0127,18.5088,0.1474
1,Planner + Memory,random,legacy6,5.25,5.3727,-0.1227,18.1115,0.1260
2,Planner Only,random,legacy6,5.44,5.3727,0.0673,19.9988,0.1306
3,Reactive Only,random,legacy6,4.32,5.3727,-1.0527,25.8572,0.2660


## Combined Ablation Scorecard

Use this table for the first read of whether the ablation results match the hypotheses. For population metrics, higher chain overlap is better and lower transition L2 is better. For persona metrics, higher anchor adherence, continuity, and delta are better; lower start-time std is better.


In [7]:
combined = (
    ABLATION_DESIGN[["setup", "use_day_planner", "memory_ablation_mode"]]
    .merge(population_summary, on="setup", how="left")
    .merge(persona_summary, on="setup", how="left")
)
combined.round(4)


,setup,use_day_planner,memory_ablation_mode,day_selector,taxonomy,avg_locations_traveled,actual_sf_avg_locations_traveled,avg_locations_delta_vs_actual,chain_weight_overlap_actual_sf_generated,transition_l2_diff_vs_actual,worker_role_anchor_adherence_mean,student_role_anchor_adherence_mean,worker_anchor_start_std_median_minutes,student_anchor_start_std_median_minutes,weekday_routine_continuity_mean,weekday_routine_similarity_delta_mean
0,Reactive Only,False,no_memory,random,legacy6,4.32,5.3727,-1.0527,25.8572,0.2660,1.0000,0.9048,25.1496,20.9162,0.6838,0.0241
1,Planner Only,True,no_memory,random,legacy6,5.44,5.3727,0.0673,19.9988,0.1306,0.9959,0.8000,47.7755,45.6886,0.6107,0.0311
2,Planner + Memory,True,no_reflection,random,legacy6,5.25,5.3727,-0.1227,18.1115,0.1260,1.0000,0.7571,34.7131,32.2878,0.6508,0.0549
3,Full Stack,True,baseline,random,legacy6,5.36,5.3727,-0.0127,18.5088,0.1474,1.0000,0.7286,20.3101,32.1113,0.7217,0.1186


## Results Interpretation

The ablation ladder should be read cumulatively: `Reactive Only` is the base system without the added components; `Planner Only` adds daily planning; `Planner + Memory` adds episodic memory without end-of-day reflection summaries; and `Full Stack` uses daily planning, memory retrieval, and reflections.

The daily planner produces the clearest population-level improvement. `Reactive Only` under-travels relative to actual SF (`avg_locations_delta_vs_actual = -1.0527`) and has the weakest transition match (`transition_l2_diff_vs_actual = 0.2660`). Adding the planner brings average locations close to actual SF (`+0.0673`) and sharply improves transition structure (`0.1306`). This supports the interpretation that daily planning gives the simulation the day-level structure needed for realistic activity/trip patterns.

`Planner + Memory` does not produce a large overall improvement over `Planner Only`. It slightly improves transition L2 (`0.1306` to `0.1260`) and improves continuity delta (`0.0311` to `0.0549`), but the gains are modest. This suggests that episodic memory by itself helps somewhat, but is not the main driver of the ablation result.

`Full Stack` is the strongest overall system. It has the closest average number of locations to actual SF (`avg_locations_delta_vs_actual = -0.0127`) and the best individual-continuity metrics (`weekday_routine_continuity_mean = 0.7217`, `weekday_routine_similarity_delta_mean = 0.1186`). The result supports the component-level claim that reflections make memory more useful by summarizing prior behavior into reusable routine context.

The daily chain-overlap metric (`chain_weight_overlap_actual_sf_generated`) appears noisy in this 100-agent random-day setting: `Reactive Only` has the highest overlap even though it performs worse on average locations and transition L2. We will revisit this metric after the 1K-agent ablation runs. If the pattern remains unstable or hard to interpret, we should remove chain overlap from the ablation argument and rely on the clearer population metrics plus the continuity metrics.


## 1K-Agent Ablation Results

This section repeats the same ablation evaluation on the completed 1K-agent OSS-120B runs. The setup, selector, taxonomy, and metric definitions match the 100-agent section above.

In [8]:
ABLATION_RUNS_1K = {
    "Reactive Only": "simulation_outputs/SF_1k_oss_120b_ablation_reactive_only",
    "Planner Only": "simulation_outputs/SF_1k_oss_120b_ablation_planner_only",
    "Planner + Memory": "simulation_outputs/SF_1k_oss_120b_ablation_planner_memory",
    "Full Stack": "simulation_outputs/SF_1k_oss_120b_ablation_full_stack",
}
INPUT_FOLDERS_1K = {
    name: "Inputs/SF_agents_1K" for name in ABLATION_RUNS_1K
}

active_runs_1k = pd.DataFrame([
    {
        "setup": setup,
        "run_folder": folder,
        "input_folder": INPUT_FOLDERS_1K[setup],
        **_activity_log_coverage(folder),
    }
    for setup, folder in ABLATION_RUNS_1K.items()
])
active_runs_1k

,setup,run_folder,input_folder,folder_exists,activity_log_files,activity_log_rows,observed_dates,date_range
0,Reactive Only,simulation_outputs/SF_1k_oss_120b_ablation_rea...,Inputs/SF_agents_1K,True,1,30162,7,2025-09-08 to 2025-09-14
1,Planner Only,simulation_outputs/SF_1k_oss_120b_ablation_pla...,Inputs/SF_agents_1K,True,1,39177,7,2025-09-08 to 2025-09-14
2,Planner + Memory,simulation_outputs/SF_1k_oss_120b_ablation_pla...,Inputs/SF_agents_1K,True,1,37376,7,2025-09-08 to 2025-09-14
3,Full Stack,simulation_outputs/SF_1k_oss_120b_ablation_ful...,Inputs/SF_agents_1K,True,1,37011,7,2025-09-08 to 2025-09-14


In [9]:
missing_1k = active_runs_1k.loc[~active_runs_1k["folder_exists"], "run_folder"].tolist()
if missing_1k:
    raise FileNotFoundError(
        "Missing real 1K ablation run folders: " + ", ".join(missing_1k)
    )

In [10]:
persona_result_1k = evaluate_models(
    ABLATION_RUNS_1K,
    INPUT_FOLDERS_1K,
    anchor_min_minutes=ANCHOR_MIN_MINUTES,
    seed=SEED,
    control_samples_per_agent_day=CONTROL_SAMPLES_PER_AGENT_DAY,
)

persona_summary_1k = (
    persona_result_1k.summary[persona_cols]
    .rename(columns={"model": "setup"})
    .round(4)
)
persona_summary_1k

,setup,worker_role_anchor_adherence_mean,student_role_anchor_adherence_mean,worker_anchor_start_std_median_minutes,student_anchor_start_std_median_minutes,weekday_routine_continuity_mean,weekday_routine_similarity_delta_mean
0,Reactive Only,1.0000,0.9863,27.7489,23.0217,0.7075,0.0263
1,Planner Only,0.9988,0.9776,59.9375,43.0116,0.6143,0.0394
2,Planner + Memory,0.9988,0.9714,30.4136,31.3050,0.6522,0.0498
3,Full Stack,0.9996,0.9739,22.5278,31.4229,0.7369,0.1269


In [11]:
with contextlib.redirect_stdout(io.StringIO()):
    reference_frames_1k = load_actual_reference_frames(target_taxonomy=TARGET_TAXONOMY)
    model_frames_1k = load_simulation_run_dict(
        ABLATION_RUNS_1K,
        day_selector=POPULATION_DAY_SELECTOR,
        seed=SEED,
        run_input_folders=INPUT_FOLDERS_1K,
        target_taxonomy=TARGET_TAXONOMY,
    )

pattern_metrics_1k = build_pattern_metrics_table(
    actual_sf_df=reference_frames_1k["actual_sf"],
    model_frames=model_frames_1k,
)
trip_metrics_1k = build_trip_metrics_table(
    actual_sf_df=reference_frames_1k["actual_sf"],
    model_frames=model_frames_1k,
)
chain_metrics_1k = build_chain_metrics_table(
    actual_sf_df=reference_frames_1k["actual_sf"],
    actual_all_df=reference_frames_1k["actual_all"],
    model_frames=model_frames_1k,
)

actual_location_mean_1k = pd.to_numeric(reference_frames_1k["actual_sf"]["location"], errors="coerce").dropna().mean()
location_mean_1k = (
    pattern_metrics_1k.loc[pattern_metrics_1k["metric"] == "location_count_mean", ["model", "value"]]
    .rename(columns={"model": "setup", "value": "avg_locations_traveled"})
)
location_mean_1k["actual_sf_avg_locations_traveled"] = actual_location_mean_1k
location_mean_1k["avg_locations_delta_vs_actual"] = (
    location_mean_1k["avg_locations_traveled"] - location_mean_1k["actual_sf_avg_locations_traveled"]
)

transition_l2_1k = (
    trip_metrics_1k.loc[trip_metrics_1k["metric"] == "transition_matrix_l2_diff_vs_actual", ["model", "value"]]
    .rename(columns={"model": "setup", "value": "transition_l2_diff_vs_actual"})
)
chain_overlap_1k = (
    chain_metrics_1k.loc[chain_metrics_1k["metric"] == "chain_weight_overlap_actual_sf_generated", ["model", "value"]]
    .rename(columns={"model": "setup", "value": "chain_weight_overlap_actual_sf_generated"})
)
population_summary_1k = (
    location_mean_1k
    .merge(chain_overlap_1k, on="setup", how="outer")
    .merge(transition_l2_1k, on="setup", how="outer")
)
population_summary_1k.insert(1, "day_selector", POPULATION_DAY_SELECTOR)
population_summary_1k.insert(2, "taxonomy", TARGET_TAXONOMY)
population_summary_1k = population_summary_1k.round(4)
population_summary_1k

,setup,day_selector,taxonomy,avg_locations_traveled,actual_sf_avg_locations_traveled,avg_locations_delta_vs_actual,chain_weight_overlap_actual_sf_generated,transition_l2_diff_vs_actual
0,Full Stack,random,legacy6,5.245,5.3727,-0.1277,24.7726,0.1436
1,Planner + Memory,random,legacy6,5.357,5.3727,-0.0157,26.3122,0.1319
2,Planner Only,random,legacy6,5.553,5.3727,0.1803,25.3672,0.1528
3,Reactive Only,random,legacy6,4.328,5.3727,-1.0447,30.0673,0.2498


In [12]:
combined_1k = (
    ABLATION_DESIGN[["setup", "use_day_planner", "memory_ablation_mode"]]
    .merge(population_summary_1k, on="setup", how="left")
    .merge(persona_summary_1k, on="setup", how="left")
)
combined_1k.round(4)

,setup,use_day_planner,memory_ablation_mode,day_selector,taxonomy,avg_locations_traveled,actual_sf_avg_locations_traveled,avg_locations_delta_vs_actual,chain_weight_overlap_actual_sf_generated,transition_l2_diff_vs_actual,worker_role_anchor_adherence_mean,student_role_anchor_adherence_mean,worker_anchor_start_std_median_minutes,student_anchor_start_std_median_minutes,weekday_routine_continuity_mean,weekday_routine_similarity_delta_mean
0,Reactive Only,False,no_memory,random,legacy6,4.328,5.3727,-1.0447,30.0673,0.2498,1.0000,0.9863,27.7489,23.0217,0.7075,0.0263
1,Planner Only,True,no_memory,random,legacy6,5.553,5.3727,0.1803,25.3672,0.1528,0.9988,0.9776,59.9375,43.0116,0.6143,0.0394
2,Planner + Memory,True,no_reflection,random,legacy6,5.357,5.3727,-0.0157,26.3122,0.1319,0.9988,0.9714,30.4136,31.3050,0.6522,0.0498
3,Full Stack,True,baseline,random,legacy6,5.245,5.3727,-0.1277,24.7726,0.1436,0.9996,0.9739,22.5278,31.4229,0.7369,0.1269


## 100-Agent vs 1K-Agent Comparison

This table keeps only the metrics used for the ablation argument so the two run sizes can be compared directly.

In [13]:
comparison_cols = [
    "setup",
    "avg_locations_traveled",
    "avg_locations_delta_vs_actual",
    "transition_l2_diff_vs_actual",
    "chain_weight_overlap_actual_sf_generated",
    "weekday_routine_continuity_mean",
    "weekday_routine_similarity_delta_mean",
]
comparison_100 = combined[comparison_cols].copy()
comparison_100.insert(1, "run_size", "100-agent")
comparison_1k = combined_1k[comparison_cols].copy()
comparison_1k.insert(1, "run_size", "1K-agent")
size_comparison = pd.concat([comparison_100, comparison_1k], ignore_index=True)
size_comparison["setup"] = pd.Categorical(size_comparison["setup"], ABLATION_DESIGN["setup"], ordered=True)
size_comparison["run_size"] = pd.Categorical(size_comparison["run_size"], ["100-agent", "1K-agent"], ordered=True)
size_comparison = size_comparison.sort_values(["setup", "run_size"]).reset_index(drop=True)
size_comparison.round(4)

,setup,run_size,avg_locations_traveled,avg_locations_delta_vs_actual,transition_l2_diff_vs_actual,chain_weight_overlap_actual_sf_generated,weekday_routine_continuity_mean,weekday_routine_similarity_delta_mean
0,Reactive Only,100-agent,4.320,-1.0527,0.2660,25.8572,0.6838,0.0241
1,Reactive Only,1K-agent,4.328,-1.0447,0.2498,30.0673,0.7075,0.0263
2,Planner Only,100-agent,5.440,0.0673,0.1306,19.9988,0.6107,0.0311
3,Planner Only,1K-agent,5.553,0.1803,0.1528,25.3672,0.6143,0.0394
4,Planner + Memory,100-agent,5.250,-0.1227,0.1260,18.1115,0.6508,0.0549
5,Planner + Memory,1K-agent,5.357,-0.0157,0.1319,26.3122,0.6522,0.0498
6,Full Stack,100-agent,5.360,-0.0127,0.1474,18.5088,0.7217,0.1186
7,Full Stack,1K-agent,5.245,-0.1277,0.1436,24.7726,0.7369,0.1269


## Updated Interpretation After 1K Run

The 1K results mostly preserve the interpretation from the 100-agent pilot, but they make the population/continuity split clearer.

`Reactive Only` remains the weakest population-structure baseline. It still under-travels by about one location per agent-day (`avg_locations_delta_vs_actual = -1.0447`) and has the worst transition match (`transition_l2_diff_vs_actual = 0.2498`). This confirms that removing daily planning leaves the system without enough day-level mobility structure.

Adding the daily planner again fixes most of that population-level problem. `Planner Only` moves average locations much closer to actual SF (`+0.1803`) and improves transition L2 substantially (`0.2498` to `0.1528`). This reinforces the claim that the daily planner is the main component for aggregate activity/trip realism.

`Planner + Memory` is the best 1K setup on the population metrics we trust most: it is closest to actual SF on average locations (`-0.0157`) and has the best transition L2 (`0.1319`). Its continuity gains over `Planner Only` are still modest (`0.0394` to `0.0498`), so memory without reflection helps, but does not by itself deliver the strongest persona-continuity effect.

`Full Stack` remains the strongest individual-continuity system. It has the highest weekday routine continuity (`0.7369`) and the largest matched-control delta (`0.1269`), very close to the 100-agent pattern. This strengthens the argument that reflection makes retrieved memory more useful for persistent routines.

The main interpretation does not change: daily planning primarily improves population-level structure, while the full memory-plus-reflection stack is needed for the strongest longitudinal persona continuity. The one wording change is that we should not claim Full Stack is best on every population metric; in the 1K run, `Planner + Memory` is better on average locations and transition L2.

Daily chain overlap should be removed or heavily de-emphasized in the ablation argument. It is highest for `Reactive Only` in both 100-agent and 1K runs, even though `Reactive Only` is clearly worse on average locations and transition L2. That makes chain overlap unstable or misleading for this component ablation.